# Convergence Analysis

Debug notebook: point to a run directory and compare **per-episode convergence** across all methods.

Shows:
- Throughput, waiting time, volunteer rate per episode (train / eval / final_eval)
- Reward & p(volunteer) convergence (MAPPO only)
- Per-agent p(volunteer) breakdown

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────

RUN_PATH = "../../results/runs/regular_fixed_simple_reward"

# Which methods to include (None = auto-discover all)
METHODS = None

# Working-time filter: only count Mon-Fri 08:00-19:00
USE_WORKING_TIME = True

# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import re
import importlib
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

import evaluate_results
importlib.reload(evaluate_results)

from evaluate_results import (
    METHOD_DISPLAY_NAMES, METHOD_ORDER,
    extract_case_metrics, extract_assignment_types, compute_volunteer_rate,
    discover_methods,
)

# ── Discover methods ──
run_dir = Path(RUN_PATH)
assert run_dir.exists(), f"Run not found: {run_dir.resolve()}"

# Detect flat vs nested structure
skip = {"logs", "configs", ".tmp_configs"}
subdirs = [d for d in run_dir.iterdir() if d.is_dir() and d.name not in skip and not d.name.startswith(".")]
has_flat = any(d.name.startswith(("mappo_", "baseline_")) for d in subdirs)
if has_flat:
    dataset_dir = run_dir
else:
    dataset_candidates = [d for d in subdirs]
    dataset_dir = dataset_candidates[0] if dataset_candidates else run_dir

discovered = discover_methods(dataset_dir)
if METHODS:
    discovered = {k: v for k, v in discovered.items() if k in METHODS}

print(f"Run: {run_dir.resolve()}")
print(f"Methods: {len(discovered)}")
for mk, rp in discovered.items():
    dn = METHOD_DISPLAY_NAMES.get(mk, mk)
    print(f"  {dn:<25} {rp}")

In [ ]:
# ── Parse per-episode data from all sources ──

def parse_episode_summaries(run_dir: Path) -> pd.DataFrame:
    """Parse episodes/episode_N/summary.txt into a DataFrame (MAPPO only)."""
    episodes_dir = run_dir / "episodes"
    if not episodes_dir.exists():
        return pd.DataFrame()
    rows = []
    for ep_dir in sorted(episodes_dir.iterdir()):
        summary_file = ep_dir / "summary.txt"
        if not summary_file.exists():
            continue
        text = summary_file.read_text()
        row = {}
        m = re.search(r'Episode\s+(\d+)', text)
        if m: row['episode'] = int(m.group(1))
        else: continue
        m = re.search(r'Total Reward:\s*([-\d.]+)', text)
        if m: row['reward'] = float(m.group(1))
        m = re.search(r'Episode Length:\s*(\d+)', text)
        if m: row['length'] = int(m.group(1))
        m = re.search(r'Time:\s*([\d.]+)\s*seconds', text)
        if m: row['time_s'] = float(m.group(1))
        m = re.search(r'Volunteer Rate:\s*([\d.]+)%\s*\((\d+)/(\d+)\)', text)
        if m:
            row['volunteer_rate'] = float(m.group(1))
            row['volunteer_count'] = int(m.group(2))
            row['total_decisions'] = int(m.group(3))
        m = re.search(r'Avg p\(volunteer\):\s*([\d.]+)', text)
        if m: row['avg_p_volunteer'] = float(m.group(1))
        for agent_match in re.finditer(r'Agent\s+(\d+):\s*([\d.]+)', text):
            agent_id = int(agent_match.group(1))
            row[f'agent_{agent_id}_p'] = float(agent_match.group(2))
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values('episode').reset_index(drop=True)


def parse_per_episode_csv_metrics(run_dir: Path, use_working_time: bool = False) -> pd.DataFrame:
    """Parse per-episode metrics from train/eval/final_eval CSV logs.
    
    Waiting time = throughput - processing (total idle/queue time per case).
    """
    rows = []
    for phase, pattern in [("train", "log_train_ep*_*.csv"),
                           ("eval", "log_eval_ep*_*.csv"),
                           ("final_eval", "log_final_eval_ep*_*.csv")]:
        log_dir = run_dir / "logs" / phase
        if not log_dir.exists():
            log_dir = run_dir / "logs"
            if not log_dir.exists():
                continue
        for csv_file in sorted(log_dir.glob(pattern)):
            m = re.search(r'ep(\d+)', csv_file.name)
            if not m:
                continue
            ep_num = int(m.group(1))
            throughput, _, processing = extract_case_metrics(
                csv_file, use_working_time=use_working_time)
            assignment_types = extract_assignment_types(csv_file)
            vol_rate = compute_volunteer_rate(assignment_types)

            # Compute waiting as throughput - processing per case
            # (total idle/queue time, not just time-to-first-task)
            if throughput and processing:
                # throughput and processing may have different lengths
                # (processing counts per-task, throughput per-case)
                # Use min length to align
                n = min(len(throughput), len(processing))
                waiting = [t - p for t, p in zip(throughput[:n], processing[:n])]
                mean_waiting = np.mean(waiting)
            else:
                mean_waiting = np.nan

            rows.append({
                'episode': ep_num,
                'phase': phase,
                'file': csv_file.name,
                'mean_throughput': np.mean(throughput) if throughput else np.nan,
                'mean_waiting': mean_waiting,
                'mean_processing': np.mean(processing) if processing else np.nan,
                'median_throughput': np.median(throughput) if throughput else np.nan,
                'volunteer_rate': vol_rate,
                'n_cases': len(throughput),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(['phase', 'episode']).reset_index(drop=True)


# ── Parse all methods ──
csv_data = OrderedDict()
summary_data = OrderedDict()

for method_name, method_dir in discovered.items():
    dn = METHOD_DISPLAY_NAMES.get(method_name, method_name)
    df_csv = parse_per_episode_csv_metrics(method_dir, use_working_time=USE_WORKING_TIME)
    if not df_csv.empty:
        csv_data[method_name] = df_csv
    df_sum = parse_episode_summaries(method_dir)
    if not df_sum.empty:
        summary_data[method_name] = df_sum
    parts = []
    if not df_sum.empty:
        parts.append(f"{len(df_sum)} episodes (summary)")
    if not df_csv.empty:
        for ph in sorted(df_csv['phase'].unique()):
            n = len(df_csv[df_csv['phase'] == ph])
            parts.append(f"{n} {ph}")
    print(f"  {dn:<25} {', '.join(parts) if parts else '(no data)'}")

In [ ]:
# ── Convergence: Train Phase ──
# 2x2 grid: Throughput, Waiting, Volunteer Rate, N Cases

def _ordered_methods(data_dict):
    """Return method keys in METHOD_ORDER, then remaining."""
    ordered = []
    for dn in METHOD_ORDER:
        for mk in data_dict:
            if METHOD_DISPLAY_NAMES.get(mk, mk) == dn and mk not in ordered:
                ordered.append(mk)
    for mk in data_dict:
        if mk not in ordered:
            ordered.append(mk)
    return ordered


def plot_phase(csv_data, phase, title_suffix=""):
    """Plot 2x2 convergence grid for a given phase."""
    ordered = _ordered_methods(csv_data)
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(ordered), 3)))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    has_data = False

    for i, mk in enumerate(ordered):
        df = csv_data[mk]
        dn = METHOD_DISPLAY_NAMES.get(mk, mk)
        ph_df = df[df['phase'] == phase].sort_values('episode')
        if ph_df.empty:
            continue
        has_data = True
        c = colors[i]
        kw = dict(marker='o', markersize=5, linewidth=2, color=c, label=dn, alpha=0.8)

        axes[0, 0].plot(ph_df['episode'], ph_df['mean_throughput'], **kw)
        axes[0, 1].plot(ph_df['episode'], ph_df['mean_waiting'], **kw)
        axes[1, 0].plot(ph_df['episode'], ph_df['volunteer_rate'], **kw)
        axes[1, 1].plot(ph_df['episode'], ph_df['n_cases'], **kw)

    if not has_data:
        plt.close(fig)
        print(f"No data for phase '{phase}'")
        return

    axes[0, 0].set_ylabel('Mean Throughput (min)')
    axes[0, 0].set_title('Mean Throughput per Episode')
    axes[0, 1].set_ylabel('Mean Waiting Time (min)')
    axes[0, 1].set_title('Mean Waiting Time per Episode (throughput − processing)')
    axes[1, 0].set_ylabel('Volunteer Rate (%)')
    axes[1, 0].set_title('Volunteer Rate per Episode')
    axes[1, 0].set_ylim(-5, 105)
    axes[1, 0].axhline(y=50, color='gray', linestyle='--', alpha=0.3)
    axes[1, 1].set_ylabel('Cases')
    axes[1, 1].set_title('Cases per Episode')

    for ax in axes.flat:
        ax.set_xlabel('Episode')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    phase_labels = {'train': 'Training', 'eval': 'Intermediate Eval', 'final_eval': 'Final Eval (test set)'}
    fig.suptitle(f"{phase_labels.get(phase, phase)} Convergence{title_suffix}",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_phase(csv_data, 'train')

In [ ]:
# ── Convergence: Eval & Final Eval Phases ──

plot_phase(csv_data, 'eval')
plot_phase(csv_data, 'final_eval')

In [ ]:
# ── Reward & p(volunteer) Convergence (MAPPO only, from episode summaries) ──

if summary_data:
    ordered = _ordered_methods(summary_data)
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(ordered), 3)))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    for i, mk in enumerate(ordered):
        df = summary_data[mk]
        dn = METHOD_DISPLAY_NAMES.get(mk, mk)
        c = colors[i]
        kw = dict(marker='o', markersize=4, linewidth=2, color=c, label=dn)

        if 'reward' in df.columns:
            axes[0, 0].plot(df['episode'], df['reward'], **kw)
        if 'volunteer_rate' in df.columns:
            axes[0, 1].plot(df['episode'], df['volunteer_rate'], **kw)
        if 'avg_p_volunteer' in df.columns:
            axes[1, 0].plot(df['episode'], df['avg_p_volunteer'], **kw)
        if 'time_s' in df.columns:
            axes[1, 1].plot(df['episode'], df['time_s'] / 60, **kw)

    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].set_title('Reward per Episode')
    axes[0, 1].set_ylabel('Volunteer Rate (%)')
    axes[0, 1].set_title('Volunteer Rate per Episode (training)')
    axes[0, 1].set_ylim(-5, 105)
    axes[0, 1].axhline(y=50, color='gray', linestyle='--', alpha=0.3)
    axes[1, 0].set_ylabel('Avg p(volunteer)')
    axes[1, 0].set_title('Average p(volunteer) per Episode')
    axes[1, 0].set_ylim(-0.05, 1.05)
    axes[1, 0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
    axes[1, 0].axhline(y=0.3, color='red', linestyle=':', alpha=0.4, label='threshold (0.3)')
    axes[1, 1].set_ylabel('Time (min)')
    axes[1, 1].set_title('Episode Duration')

    for ax in axes.flat:
        ax.set_xlabel('Episode')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    fig.suptitle('Training Convergence (from episode summaries)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No episode summary data (only MAPPO methods have these).')

In [ ]:
# ── Per-Agent p(volunteer) Breakdown ──
# One subplot per MAPPO method, one line per agent

if summary_data:
    ordered = _ordered_methods(summary_data)
    n_plots = len(ordered)
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5), squeeze=False)
    agent_colors = plt.cm.tab10(np.linspace(0, 1, 10))

    for idx, mk in enumerate(ordered):
        ax = axes[0, idx]
        df = summary_data[mk]
        dn = METHOD_DISPLAY_NAMES.get(mk, mk)

        agent_cols = sorted([c for c in df.columns if c.startswith('agent_') and c.endswith('_p')])
        for i, col in enumerate(agent_cols):
            agent_id = col.replace('agent_', '').replace('_p', '')
            ax.plot(df['episode'], df[col], marker='o', markersize=3,
                    label=f'Agent {agent_id}', color=agent_colors[i],
                    linewidth=1.5, alpha=0.8)

        if 'avg_p_volunteer' in df.columns:
            ax.plot(df['episode'], df['avg_p_volunteer'], 'k--',
                    linewidth=2, label='Average', alpha=0.7)

        ax.set_xlabel('Episode')
        ax.set_ylabel('p(volunteer)')
        ax.set_title(f'{dn} — Per-Agent p(volunteer)')
        ax.set_ylim(-0.05, 1.05)
        ax.axhline(y=0.3, color='red', linestyle=':', alpha=0.4, label='threshold (0.3)')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No per-agent data (only MAPPO methods have these).')